In [0]:
VECTOR_DB_PATH = "/Volumes/workspace/legal_data/vector_db/"

In [0]:
%pip install sentence-transformers chromadb

In [0]:
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

In [0]:
GOLD_PATH = "/Volumes/workspace/legal_data/gold/legal_chunks/"

gold_df = spark.read.format("delta").load(GOLD_PATH)

gold_df = gold_df.select(
    "chunk_id",
    "chunk_text",
    "act_name",
    "section_number",
    "category",
    "file_name"
)

gold_df.display()

In [0]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [0]:
VECTOR_DB_PATH = "/local_disk0/tmp/chroma_db"

/Volumes/workspace/legal_data/vector_db/

In [0]:
import os

os.makedirs(VECTOR_DB_PATH, exist_ok=True)

In [0]:
import chromadb

client = chromadb.PersistentClient(path=VECTOR_DB_PATH)

collection = client.get_or_create_collection(
    name="legal_knowledge",
    metadata={"hnsw:space": "cosine"}
)

print("Vector DB ready")

In [0]:
def safe_str(value):
    if value is None:
        return ""
    return str(value)

In [0]:
rows = gold_df.toLocalIterator()

In [0]:
batch_size = 100
batch = []

for row in rows:
    batch.append(row)

    if len(batch) == batch_size:
        texts = [safe_str(r.chunk_text) for r in batch]
        ids = [safe_str(r.chunk_id) for r in batch]

        embeddings = model.encode(texts).tolist()

        metadata = []
        for r in batch:
            metadata.append({
                "act_name": safe_str(r.act_name),
                "section": safe_str(r.section_number),
                "category": safe_str(r.category),
                "source": safe_str(r.file_name),
            })

        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadata
        )

        print(f"Inserted {len(batch)} records")
        batch = []

# insert remaining rows
if batch:
    texts = [safe_str(r.chunk_text) for r in batch]
    ids = [safe_str(r.chunk_id) for r in batch]
    embeddings = model.encode(texts).tolist()

    metadata = []
    for r in batch:
        metadata.append({
            "act_name": safe_str(r.act_name),
            "section": safe_str(r.section_number),
            "category": safe_str(r.category),
            "source": safe_str(r.file_name),
        })

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadata
    )

    print(f"Inserted final {len(batch)} records")

In [0]:
collection.count()

In [0]:
query = "What is the penalty under Motor Vehicles Act for not wearing helmet under section 129?"

query_embedding = model.encode([query]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=10
)

docs = results["documents"][0]

filtered_docs = [doc for doc in docs if "helmet" in doc.lower()]

print(filtered_docs[:3])